# 📝 Data Drift — Complete Notes

---

## 🤔 What is Data Drift?

Data Drift means the **statistical properties of data change over time**

- The data your model was **trained on** looks different from the **new incoming data**
- This causes your ML model to give **wrong predictions**

---

## 💡 Real Life Example

> Imagine you trained a **spam email detector** in 2020
> - In 2020, spam emails used words like *"Free Money"*, *"Click Here"*
> - In 2024, spammers changed their language completely
> - Now your model **fails to detect** new spam emails
> - This is **Data Drift!** 📉

---

## 🆚 Types of Drift

| Type | Meaning | Example |
|---|---|---|
| **Feature Drift** | Input data distribution changes | Age range of users changed |
| **Label Drift** | Output/target distribution changes | More fraud cases than before |
| **Concept Drift** | Relationship between input & output changes | What was spam before is not spam now |

---

## 📊 How Do We Detect Data Drift?

We use **KS Test (Kolmogorov-Smirnov Test)**

```python
from scipy.stats import ks_2samp
```

### What does `ks_2samp` do?
- Compares **two distributions** (train vs test)
- Returns a **p-value**
- If `p-value < 0.05` → **Drift detected** ⚠️
- If `p-value >= 0.05` → **No drift** ✅

---

## 📝 Data Drift Detection Code With Comments

```python
def detect_dataset_drift(self, base_df, current_df, threshold=0.05) -> bool:
    try:
        # Start drift detection
        logging.info("Checking for data drift...")

        # Flag to track if drift was found
        status = True

        # Dictionary to store drift report for each column
        report = {}

        # Loop through each column in the dataframe
        for column in base_df.columns:

            # Get the column data from train dataframe (base)
            d1 = base_df[column]

            # Get the column data from test dataframe (current)
            d2 = current_df[column]

            # Apply KS Test to compare both distributions
            # ks_2samp returns:
            # is_same_dist → test statistic (how different they are)
            # p_value → probability that both come from same distribution
            is_same_dist = ks_2samp(d1, d2)

            # If p_value < threshold (0.05) → distributions are different → Drift!
            if threshold <= is_same_dist.pvalue:
                is_found = False  # No drift found for this column
            else:
                is_found = True   # Drift found for this column ⚠️
                status = False    # Overall status becomes False

            # Store the result for this column in report
            report.update({
                column: {
                    "p_value": float(is_same_dist.pvalue),  # probability value
                    "drift_status": is_found                 # True = drift, False = no drift
                }
            })

        # Path where drift report will be saved
        drift_report_file_path = self.data_validation_config.drift_report_file_path

        # Create the directory if it doesn't exist
        dir_path = os.path.dirname(drift_report_file_path)
        os.makedirs(dir_path, exist_ok=True)

        # Save the drift report as a YAML file
        write_yaml_file(
            file_path=drift_report_file_path,
            content=report
        )

        # Return overall drift status
        # True = No drift, False = Drift detected
        return status

    except Exception as e:
        raise CustomException(e, sys)
```

---

## 📊 What the Drift Report Looks Like

```yaml
# drift_report.yaml
having_IP_Address:
  p_value: 0.923        # High p_value = No drift ✅
  drift_status: False

URL_Length:
  p_value: 0.003        # Low p_value = Drift detected ⚠️
  drift_status: True

Shortining_Service:
  p_value: 0.071        # p_value > 0.05 = No drift ✅
  drift_status: False
```

---

## 🔁 How p_value Works

```
p_value >= 0.05  →  Same distribution  →  No Drift  ✅
p_value <  0.05  →  Different distribution  →  Drift Detected  ⚠️
```

---

## 💡 Simple Analogy
> Think of KS Test like **comparing two water samples** 🧪
> - Sample 1 = Train data (old water)
> - Sample 2 = Test data (new water)
> - KS Test checks if both samples have **same composition**
> - If very different → **something changed** → Drift! ⚠️

---

## 🔁 Full Data Validation Flow

```
Load Train & Test Data
        ↓
Validate Number of Columns ✅ or ❌
        ↓
Validate Numerical Columns ✅ or ❌
        ↓
Run KS Test on each column
        ↓
p_value < 0.05 → Drift ⚠️
p_value ≥ 0.05 → No Drift ✅
        ↓
Save Drift Report (YAML file)
        ↓
Return DataValidationArtifact
```

---

## ⚡ One Line Summary
> *Data Drift = your model's training data no longer represents real world data, detected using KS Test by comparing p_values* 🚀